# 04c - Latent Propagator Comparison

Train one shared latent autoencoder, then compare different latent propagators on the same encoded trajectories.

The baseline propagator uses the historical normalized `dz` objective. The JEPA-style option predicts the next latent embedding directly (`z(t+1)`) and is evaluated by autoregressive rollout through the same decoder.


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lss.latent.comparison import run_propagator_comparison
from lss.latent.experiment import seed_everything
from lss.utils import resolve_device

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})


## Configuration


In [ ]:
seed = random.SystemRandom().randrange(1, 2**31)
seed_everything(seed)
print('run seed:', seed)

selected_dataset = 'depablo'  # 'reid' or 'depablo'

dataset_specs = {
    'reid': {
        'label': 'Reid',
        'path': '../../data/new_reid_combined.pt',
        'train_count': 20,
        'val_count': 20,
    },
    'depablo': {
        'label': 'dePablo OOL',
        'path': '../../data/2340_dePablo_networks_OOL_undirected.pt',
        'train_count': 10,
        'val_count': 25,
    },
}

cfg = {
    'dataset_name': selected_dataset,
    'split_seed': seed,
    'device': 'auto',
    'pos_dim': 2,
    'batch_graphs': 8,
    'frame_skip': 1,
    'train_frame_start_order': 1,
    'latent_dim': 4,
    'latent_tokens': 8,
    'hidden_size': 90,
    'ae_train_frames_per_sim': 30,
    'dyn_train_transitions_per_sim': 30,
    'ae_target_mode': 'delta',
    'node_feature_mode': 'delta',
    'ae_max_epochs': 250,
    'ae_patience': 8,
    'ae_lr': 2e-4,
    'ae_weight_decay': 1e-5,
    'propagator_max_epochs': 250,
    'propagator_patience': 8,
    'propagator_lr': 2e-4,
    'propagator_weight_decay': 1e-5,
    'multistep_horizons': [1, 5, 10, 20],
    'multistep_max_starts_per_sim': 50,
    'early_stop_min_delta': 1e-5,
    'rollout_steps_grid': list(range(10, 301, 10)),
    'p_ratio_methods': ['rollout_sides', 'box', 'rollout_all', 'rollout_outer'],
    'force_train_ae': True,
    'force_train_propagators': True,
    'propagator_repeats': 3,
    'output_dir': Path('../results/latent_propagator_comparison') / selected_dataset,
}

propagator_specs = [
    # {'name': 'delta_residual_mlp90', 'loss_mode': 'delta', 'model_type': 'residual_mlp', 'train_objective': 'one_step', 'hidden_size': 90},
    # {'name': 'delta_residual_mlp32', 'loss_mode': 'delta', 'model_type': 'residual_mlp', 'train_objective': 'one_step', 'hidden_size': 32},
    {'name': 'delta_residual_mlp8', 'loss_mode': 'delta', 'model_type': 'residual_mlp', 'train_objective': 'one_step', 'hidden_size': 8},
     {'name': 'delta_residual_mlp2', 'loss_mode': 'delta', 'model_type': 'residual_mlp', 'train_objective': 'one_step', 'hidden_size': 2},
    # {'name': 'delta_linear', 'loss_mode': 'delta', 'model_type': 'linear_residual', 'train_objective': 'one_step'},
    # {'name': 'delta_hybrid', 'loss_mode': 'hybrid_delta_next', 'model_type': 'residual_mlp', 'train_objective': 'one_step', 'next_loss_weight': 0.1},
    # {'name': 'delta_velocity', 'loss_mode': 'velocity_delta', 'model_type': 'velocity_mlp', 'train_objective': 'velocity'},
    # {'name': 'delta_multistep', 'loss_mode': 'delta', 'model_type': 'residual_mlp', 'train_objective': 'multistep'},
    # {'name': 'jepa_next_z', 'loss_mode': 'next_z', 'model_type': 'direct_mlp', 'train_objective': 'one_step'},
]

cfg['output_dir'].mkdir(parents=True, exist_ok=True)
device = resolve_device(cfg['device'])
print('device:', device)
print('output:', cfg['output_dir'])







## Train Or Load Shared AE And Propagators


In [ ]:
source_spec = dataset_specs[cfg['dataset_name']]
comparison = run_propagator_comparison(
    source_spec,
    cfg,
    propagator_specs,
    seed=seed,
    device=device,
)

train_data = comparison['train_data']
val_data = comparison['val_data']
test_data = comparison['test_data']
ae_history = comparison['ae_history']
prop_history_df = comparison['propagator_history']
rollout_raw_df = comparison['rollout_raw']
rollout_stats_df = comparison['rollout_stats']
cfg['rollout_steps_grid'] = comparison['rollout_steps_grid']

print('split sizes:', len(train_data), len(val_data), len(test_data))
print('rollout horizons:', cfg['rollout_steps_grid'])
display(rollout_stats_df[rollout_stats_df['split'].eq('test')].round(6))


## Comparison


In [ ]:
# Plots: mean ± std over propagator repeats.
test_stats = rollout_stats_df[rollout_stats_df['split'].eq('test')].copy()
curve_df = (
    test_stats
    .groupby(['p_ratio_method', 'propagator', 'rollout_steps'], as_index=False)
    .agg(
        p_ratio_r2_mean=('p_ratio_r2', 'mean'),
        p_ratio_r2_std=('p_ratio_r2', 'std'),
        final_pos_mse_mean=('final_pos_mse', 'mean'),
        final_pos_mse_std=('final_pos_mse', 'std'),
        n_repeats=('repeat_idx', 'nunique'),
    )
)

methods = list(cfg['p_ratio_methods'])
fig, axes = plt.subplots(len(methods), 2, figsize=(11.8, 3.3 * len(methods)), constrained_layout=True, squeeze=False)
for row_idx, p_ratio_method in enumerate(methods):
    method_curve = curve_df[curve_df['p_ratio_method'].eq(p_ratio_method)].copy()
    for name, group in method_curve.groupby('propagator'):
        group = group.sort_values('rollout_steps')
        x = group['rollout_steps'].to_numpy(dtype=float)
        y = group['p_ratio_r2_mean'].to_numpy(dtype=float)
        yerr = group['p_ratio_r2_std'].fillna(0).to_numpy(dtype=float)
        axes[row_idx, 0].plot(x, y, marker='o', lw=2, ms=3, label=name)
        axes[row_idx, 0].fill_between(x, y - yerr, y + yerr, alpha=0.12)

        mse = group['final_pos_mse_mean'].to_numpy(dtype=float)
        mse_std = group['final_pos_mse_std'].fillna(0).to_numpy(dtype=float)
        lower = np.maximum(mse - mse_std, 1e-12)
        upper = mse + mse_std
        axes[row_idx, 1].plot(x, mse, marker='o', lw=2, ms=3, label=name)
        axes[row_idx, 1].fill_between(x, lower, upper, alpha=0.12)
    axes[row_idx, 0].axhline(0, color='0.25', lw=0.9)
    axes[row_idx, 0].set_ylim(-0.05, 1.02)
    axes[row_idx, 0].set_xlabel('rollout steps')
    axes[row_idx, 0].set_ylabel('test p-ratio R2, mean ± std')
    axes[row_idx, 0].set_title(f'property rollout: {p_ratio_method}')
    axes[row_idx, 1].set_yscale('log')
    axes[row_idx, 1].set_xlabel('rollout steps')
    axes[row_idx, 1].set_ylabel('test position MSE, mean ± std')
    axes[row_idx, 1].set_title(f'geometry error: {p_ratio_method}')
    for ax in axes[row_idx]:
        ax.legend(frameon=False, fontsize=7)
fig.suptitle(f"{source_spec['label']}: shared AE, propagator comparison ({cfg['propagator_repeats']} repeats each)", fontsize=12, fontweight='bold')
plt.show()

summary_by_repeat = (
    test_stats.groupby(['p_ratio_method', 'propagator', 'repeat_idx'], as_index=False)
    .agg(
        mean_p_ratio_r2=('p_ratio_r2', 'mean'),
        min_p_ratio_r2=('p_ratio_r2', 'min'),
        step100_p_ratio_r2=('p_ratio_r2', lambda s: float(test_stats.loc[s.index][test_stats.loc[s.index, 'rollout_steps'].eq(100)]['p_ratio_r2'].iloc[0]) if any(test_stats.loc[s.index, 'rollout_steps'].eq(100)) else np.nan),
        step200_p_ratio_r2=('p_ratio_r2', lambda s: float(test_stats.loc[s.index][test_stats.loc[s.index, 'rollout_steps'].eq(200)]['p_ratio_r2'].iloc[0]) if any(test_stats.loc[s.index, 'rollout_steps'].eq(200)) else np.nan),
        mean_position_mse=('final_pos_mse', 'mean'),
        step100_position_mse=('final_pos_mse', lambda s: float(test_stats.loc[s.index][test_stats.loc[s.index, 'rollout_steps'].eq(100)]['final_pos_mse'].iloc[0]) if any(test_stats.loc[s.index, 'rollout_steps'].eq(100)) else np.nan),
    )
)
summary = (
    summary_by_repeat.groupby(['p_ratio_method', 'propagator'], as_index=False)
    .agg(
        mean_p_ratio_r2=('mean_p_ratio_r2', 'mean'),
        std_p_ratio_r2=('mean_p_ratio_r2', 'std'),
        min_p_ratio_r2_mean=('min_p_ratio_r2', 'mean'),
        step100_p_ratio_r2_mean=('step100_p_ratio_r2', 'mean'),
        step100_p_ratio_r2_std=('step100_p_ratio_r2', 'std'),
        step200_p_ratio_r2_mean=('step200_p_ratio_r2', 'mean'),
        step200_p_ratio_r2_std=('step200_p_ratio_r2', 'std'),
        mean_position_mse=('mean_position_mse', 'mean'),
        std_position_mse=('mean_position_mse', 'std'),
        n_repeats=('repeat_idx', 'nunique'),
    )
    .sort_values(['p_ratio_method', 'mean_p_ratio_r2'], ascending=[True, False])
)
print('aggregate over repeats')
display(summary.round(6))
print('per-repeat summary')
display(summary_by_repeat.sort_values(['p_ratio_method', 'propagator', 'repeat_idx']).round(6))
